# Day 19 – Hypothesis Testing
## Turning Data Observations into Statistical Evidence

**30 Days of Data Analytics – M.V. Krishnakumari**

This notebook demonstrates hypothesis testing using a public-grievance-style example.

### Learning objectives
- Define null and alternative hypotheses
- Understand significance level and p-value
- Perform Welch's independent two-sample t-test
- Calculate a confidence interval
- Examine effect size
- Distinguish statistical significance from practical significance
- Understand why significance does not automatically imply causation

## 1. Business Question

Suppose a grievance department introduces a new workflow.

Before implementation, average complaint resolution time is about **52 hours**.  
After implementation, it is about **47 hours**.

**Question:** Is the observed reduction statistically significant, or could it be normal variation?

### Hypotheses
- **H₀:** There is no difference in mean resolution time.
- **H₁:** There is a difference in mean resolution time.

We use **α = 0.05**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

In [ ]:
before = np.random.normal(loc=52, scale=12, size=300)
after = np.random.normal(loc=47, scale=12, size=300)

before = np.maximum(before, 0)
after = np.maximum(after, 0)

## 2. Descriptive Comparison

In [ ]:
summary = pd.DataFrame({
    "Period": ["Before", "After"],
    "Mean Resolution Time": [before.mean(), after.mean()],
    "Median Resolution Time": [np.median(before), np.median(after)],
    "Std Dev": [before.std(ddof=1), after.std(ddof=1)],
    "Records": [len(before), len(after)]
})
summary

In [ ]:
difference = before.mean() - after.mean()

print(f"Before mean: {before.mean():.2f} hours")
print(f"After mean:  {after.mean():.2f} hours")
print(f"Observed reduction: {difference:.2f} hours")

## 3. Visualize the Groups

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot([before, after], labels=["Before", "After"])
plt.ylabel("Resolution Time (Hours)")
plt.title("Resolution Time Before vs After")
plt.grid(axis="y", alpha=0.25)
plt.show()

## 4. Welch's Independent Two-Sample t-test

We use Welch's t-test with `equal_var=False`, which does not assume equal population variances.

In [ ]:
result = stats.ttest_ind(
    before,
    after,
    equal_var=False
)

print(f"T-statistic: {result.statistic:.4f}")
print(f"P-value:     {result.pvalue:.6f}")

## 5. Statistical Decision

Using **α = 0.05**:

- If p-value < 0.05 → Reject H₀
- If p-value ≥ 0.05 → Fail to reject H₀

In [ ]:
alpha = 0.05

if result.pvalue < alpha:
    print("Decision: Reject H₀")
    print("There is statistically significant evidence of a difference in mean resolution time.")
else:
    print("Decision: Fail to reject H₀")
    print("There is not sufficient statistical evidence of a difference in mean resolution time.")

## 6. 95% Confidence Interval

In [ ]:
ci = result.confidence_interval(confidence_level=0.95)

print(f"95% CI lower bound: {ci.low:.4f}")
print(f"95% CI upper bound: {ci.high:.4f}")

## 7. Effect Size – Cohen's d

Statistical significance does not tell us how large the effect is. Cohen's d provides a standardized measure of the difference between means.

In [ ]:
pooled_std = np.sqrt(
    ((len(before) - 1) * np.var(before, ddof=1) +
     (len(after) - 1) * np.var(after, ddof=1))
    / (len(before) + len(after) - 2)
)

cohens_d = (before.mean() - after.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.4f}")

### Rough Cohen's d guidelines

- 0.2 → small effect
- 0.5 → medium effect
- 0.8 → large effect

These are guidelines, not universal rules. Domain context matters.

## 8. Statistical vs Practical Significance

A statistically significant result may still be operationally unimportant.

Consider:
- p-value → statistical evidence
- confidence interval → uncertainty
- effect size → magnitude
- domain context → operational importance

For public grievance analytics, also consider SLA targets, complaint volume, complaint severity, staffing, workload, and seasonality.

## 9. Significance Is Not Causation

Even a statistically significant difference does not automatically prove that the new workflow caused the improvement.

Other factors may have changed:
- Staffing levels
- Complaint mix
- Seasonal workload
- Policy changes
- Department processes
- Data-entry practices

Causal conclusions require appropriate study design and additional evidence.

## 10. Common Hypothesis Tests

| Question | Possible Test |
|---|---|
| Sample mean vs target | One-sample t-test |
| Two independent means | Independent t-test / Welch's t-test |
| Paired before-after observations | Paired t-test |
| Relationship between categorical variables | Chi-square test |
| Compare more than two group means | ANOVA |

Start with the analytical question and study design, then select the test.

## 11. Public Grievance Applications

Hypothesis testing can help answer questions such as:

1. Did average resolution time change after a process intervention?
2. Is resolution time significantly different between departments?
3. Is escalation associated with complaint category?
4. Are resolution times different across zones?
5. Did complaint volume change significantly after a policy change?

The correct statistical method depends on the question, data type, and study design.

## 12. Key Takeaways

- A difference in the data is not automatically meaningful.
- Define H₀ and H₁ before testing.
- Choose the statistical test according to the study design.
- Interpret the p-value correctly.
- Do not treat p < 0.05 as proof.
- Consider confidence intervals and effect size.
- Statistical significance is different from practical significance.
- Statistical significance does not establish causation.
- Connect statistical evidence to the operational decision.

### Final thought

**Data can show us that something changed. Statistics helps us determine whether the evidence supports that observation.**

**Data → Evidence → Better Decisions**

### Next
**Day 20 – Confidence Intervals**

GitHub Repository:
https://github.com/krishnakumarimv/30-Days-Data-Analytics-Cookbook